# METEOR-Net: Google Colab T4 GPU Training Workflow

**Project:** Flow-free Temporal Super-Resolution of Geostationary Satellite Cloud Observations  
**Target Architecture:** METEOR-Net (Lin et al., IEEE TGRS 2026)  
**Hardware Target:** Google Colab with NVIDIA T4 GPU (16 GB VRAM)  
**Baseline Checkpoint Reference:** Phase 7B CPU Baseline (Epoch 5, Val MAE: 10.1650 K)  
**Active Experiment ID:** `colab_t4_phase8` (Stored under `outputs/colab_t4_phase8/`, leaving `outputs/phase7b/` 100% untouched)

---
### Pipeline Safeguards Enforced:
1. **Partition Isolation:** Training uses strictly Events 1 & 2 (Jan 1 & May 20, 2026). Validation uses Event 3 (July 19, 2026). The September 7, 2026 test event is **strictly quarantined**.
2. **Normalization Invariance:** Frozen training-only ChannelStandardizer statistics are verified bit-for-bit against SHA-256 `71ef64...`.
3. **Physical Kelvin Precision:** All inputs, loss thresholds (241 K, 221 K), and evaluations operate in true physical Kelvin under FP32 precision.
4. **Atomic Drive Persistence:** Checkpoints and logs sync to persistent Google Drive storage to survive session timeouts.

## Stage 1: Explicit Google Drive Authorization & Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
drive_data_dir = "/content/drive/MyDrive/meteor_net_data"
drive_run_dir = "/content/drive/MyDrive/meteor_net_runs"
os.makedirs(drive_data_dir, exist_ok=True)
os.makedirs(drive_run_dir, exist_ok=True)
print(f"[Drive Mounted] Data directory: {drive_data_dir}")
print(f"[Drive Mounted] Runs directory: {drive_run_dir}")

## Stage 2: Workspace Setup & Repository Ingestion
Clone the verified codebase from the private repository (or unpack a versioned source archive into `/content/meteor_net`).

In [ ]:
# If using private GitHub repository, you can provide a GitHub Personal Access Token (PAT)
# Or copy a pre-packaged source archive from Drive
import os, subprocess

%cd /content
workspace_dir = "/content/meteor_net"

# Check if source bundle exists on Drive, otherwise clone
drive_source_zip = "/content/drive/MyDrive/meteor_net_data/meteor_net_source_bundle.zip"
if os.path.exists(drive_source_zip):
    print(f"Extracting source bundle from Drive: {drive_source_zip}")
    !unzip -q {drive_source_zip} -d /content/meteor_net
else:
    print("Cloning repository from GitHub...")
    # Note: Replace <GITHUB_TOKEN> if prompted for authentication on private repo
    !git clone https://github.com/KrishnaManoharReddy907/meteor-net-insat3ds.git /content/meteor_net

%cd /content/meteor_net
!git status
!git log -1 --oneline

## Stage 3: Dataset Ingestion onto High-Speed VM Local Disk
Unpack the 4-event GOES-19 dataset archive (`data_goes19_4events.tar.gz`) from Google Drive directly into the Colab VM's local high-speed disk (`/content/meteor_net/data`).

In [ ]:
import os, subprocess

archive_path = "/content/drive/MyDrive/meteor_net_data/data_goes19_4events.tar.gz"
assert os.path.exists(archive_path), f"Dataset archive not found at: {archive_path}. Please upload it first."

print(f"Dataset archive found ({os.path.getsize(archive_path) / (1024**3):.2f} GB). Extracting to local VM storage...")
!tar -xzf {archive_path} -C /content/meteor_net/

# Verify file count (must be exactly 1,152 NetCDF files)
nc_count = sum(len(files) for _, _, files in os.walk('/content/meteor_net/data') if any(f.endswith('.nc') for f in files))
print(f"Extracted NetCDF file count: {nc_count} / 1152")
assert nc_count == 1152, f"Incomplete dataset extraction: expected 1152, found {nc_count}"

## Stage 4: Install Dependencies & Hardware Preflight Audit

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    free_vram, total_vram = torch.cuda.mem_get_info()
    print(f"GPU VRAM: {free_vram / (1024**3):.2f} GB free / {total_vram / (1024**3):.2f} GB total")
!nvidia-smi

## Stage 5: Cryptographic Integrity & Normalization Provenance Check

In [ ]:
import hashlib, json

def verify_sha256(filepath, expected_hash):
    with open(filepath, "rb") as f:
        actual_hash = hashlib.sha256(f.read()).hexdigest()
    assert actual_hash == expected_hash, f"SHA-256 mismatch for {filepath}:\n  Got:      {actual_hash}\n  Expected: {expected_hash}"
    print(f"[VERIFIED] {filepath}\n  SHA-256: {actual_hash}")

print("Verifying critical manifest hashes...")
verify_sha256("outputs/dataset_splits/splits_manifest.json", "6a173db4760c49639f4b830f6814f97d2d970a1da48c9d62e19c81d20bbd7d7f")
verify_sha256("outputs/normalization/frozen_global_train_stats.json", "71ef64929a8875490a24cbd33e8aec87e14349cb966d32d0f68a28abc51c4164")
verify_sha256("outputs/full_acquisition/acquisition_inventory.json", "26cf302690ce3877d6e8923e5ea4141304447a5e81065adfc244e9c8ff6b4de7")

# Check test event quarantine
with open("outputs/dataset_splits/splits_manifest.json", "r") as f:
    manifest = json.load(f)
assert manifest["evaluation_validity"]["group_to_split_mapping"]["event_2026-09-07"] == "test"
print("[VERIFIED] Event 4 (September 7, 2026) is strictly quarantined as 'test'.")

## Stage 6: Fast GPU Smoke Test (Preflight Validation)
Runs `scripts/run_gpu_smoke_test.py` to exercise the real model, frozen normalization, loss, forward pass, autograd, optimizer step, and checkpoint round-trip on the T4 GPU before starting any multi-epoch training.

In [ ]:
!python scripts/run_gpu_smoke_test.py \
    --device cuda \
    --output_dir outputs/colab_t4_smoke_test

import json
with open("outputs/colab_t4_smoke_test/gpu_smoke_test_results.json", "r") as f:
    smoke_summary = json.load(f)
print("\n--- GPU Smoke Test Results ---")
print(json.dumps(smoke_summary, indent=2))
assert smoke_summary["status"] == "PASS", "GPU Smoke Test FAILED! Do not proceed to full training."

## Stage 7: Full Controlled Training Runner (Phase 8 GPU Experiment)
Target directory: `outputs/colab_t4_phase8` (Phase 7B CPU baseline remains 100% frozen).

If resuming an interrupted session, add `--resume outputs/colab_t4_phase8/latest_checkpoint.pt`.

In [ ]:
!python scripts/train_meteor_net.py \
    --device cuda \
    --output_dir outputs/colab_t4_phase8 \
    --epochs 5 \
    --microbatches_per_epoch 100 \
    --grad_accum_steps 2 \
    --lr 1e-4 \
    --weight_decay 1e-4 \
    --seed 42 \
    --val_patches 25 \
    --early_stopping_patience 2

## Stage 8: Sync Checkpoints and Training Artifacts to Google Drive

In [ ]:
import os, shutil, glob

src_dir = "outputs/colab_t4_phase8"
dest_dir = "/content/drive/MyDrive/meteor_net_runs/colab_t4_phase8"
os.makedirs(dest_dir, exist_ok=True)

# Atomic copy to Drive (copy to .tmp then rename)
for filename in os.listdir(src_dir):
    src_file = os.path.join(src_dir, filename)
    if os.path.isfile(src_file):
        dest_file = os.path.join(dest_dir, filename)
        dest_tmp = dest_file + ".tmp"
        shutil.copy2(src_file, dest_tmp)
        os.replace(dest_tmp, dest_file)
        print(f"Synced to Drive: {filename} ({os.path.getsize(dest_file)/(1024*1024):.2f} MB)")

print("\nAll Colab T4 Phase 8 experiment artifacts successfully backed up to persistent Google Drive.")